# 0. 开始

In [1]:
%load_ext autoreload
%autoreload 2

In [12]:
import os
import utils_z
import geopandas as gpd
import shp_parser as shpar

In [3]:
conn = utils_z.get_conn("Test20260413", "postgres", "we6666", "localhost", "5432")

In [4]:
city_name = "toronto"
city_prefix  = "CA_TO"

# 1. 处理 shapefile 数据并导入数据库

## 1.1 查看数据

In [ ]:
# 先看解压后有哪些文件
test_shp_dir = r"E:\2_data\building_3d_opensource\toronto"
for f in os.listdir(test_shp_dir):
    print(f)

3DMassingShapefile_2025_WGS84.cpg
3DMassingShapefile_2025_WGS84.dbf
3DMassingShapefile_2025_WGS84.prj
3DMassingShapefile_2025_WGS84.sbn
3DMassingShapefile_2025_WGS84.sbx
3DMassingShapefile_2025_WGS84.shp
3DMassingShapefile_2025_WGS84.shp.xml
3DMassingShapefile_2025_WGS84.shx
3DMassingShapefile_2025_WGS84.zip.zip
README_Metadata.xlsx


In [ ]:
test_shp_path = rf"E:\2_data\building_3d_opensource\{city_name}\shp\103082_bkm.shp"
gdf = gpd.read_file(test_shp_path)

# 基本信息
print(f"CRS: {gdf.crs}")
print(f"总行数: {len(gdf)}")
print(f"字段名: {list(gdf.columns)}")
print(f"geometry类型: {gdf.geometry.type.unique()}")

# 前几行
print(gdf.head())

CRS: EPSG:3857
总行数: 428184
字段名: ['MIN_HEIGHT', 'MAX_HEIGHT', 'AVG_HEIGHT', 'HEIGHT_MSL', 'SURF_ELEV', 'HEIGHT_SRC', 'BLDG_SRC', 'LONGITUDE', 'LATITUDE', 'geometry']
geometry类型: ['Polygon' 'MultiPolygon']
   MIN_HEIGHT  MAX_HEIGHT  AVG_HEIGHT  HEIGHT_MSL   SURF_ELEV     HEIGHT_SRC  \
0         0.0      0.0000      3.1000    171.6673  168.567290      Site Plan   
1         0.0      8.5916      5.6926    171.9050  166.212195  Lidar-Derived   
2         0.0      7.3417      4.2834    172.3670  168.083121  Lidar-Derived   
3         0.0      7.9916      5.2575    173.4900  168.232606  Lidar-Derived   
4         0.0      9.6221      5.3149    175.2660  169.951113  Lidar-Derived   

           BLDG_SRC  LONGITUDE   LATITUDE  \
0         Site Plan -79.620546  43.722195   
1  Photogrammetrics -79.616530  43.723514   
2  Photogrammetrics -79.603956  43.730254   
3  Photogrammetrics -79.620705  43.734190   
4  Photogrammetrics -79.618915  43.720103   

                                            

In [7]:
# 检查有没有高度相关字段
print(gdf.describe())

          MIN_HEIGHT     MAX_HEIGHT     AVG_HEIGHT     HEIGHT_MSL  \
count  428184.000000  428184.000000  428184.000000  428184.000000   
mean        0.586183      10.481655       7.211568     147.828734   
std         3.519083       7.186416       9.946164      33.857322   
min         0.000000       0.000000       0.002400       0.000000   
25%         0.000000       7.128475       4.260700     125.351000   
50%         0.000000       9.096800       5.543700     151.710665   
75%         0.000000      12.262000       7.036400     172.705668   
max       130.950000     251.920000     443.000000     522.684000   

           SURF_ELEV      LONGITUDE       LATITUDE  
count  428184.000000  428184.000000  428184.000000  
mean      141.293258     -79.390192      43.716081  
std        32.540291       0.110778       0.054064  
min         0.000000     -79.637468      43.585740  
25%       118.098777     -79.476959      43.674336  
50%       145.433050     -79.399488      43.710735  
75%    

In [8]:
print(f"MAX_HEIGHT为0的建筑: {(gdf['MAX_HEIGHT'] == 0).sum()}")
print(f"占比: {(gdf['MAX_HEIGHT'] == 0).mean():.1%}")

MAX_HEIGHT为0的建筑: 18481
占比: 4.3%


In [9]:
print(gdf['geometry'].geom_type.value_counts())

Polygon         427963
MultiPolygon       221
Name: count, dtype: int64


## 1.2 变量

In [ ]:
block_table_name = f"block.{city_name}_blocks"

lod1_table_name_full = f"lod1.{city_name}_buildings_lod1"
lod1_surface_table_name_full = f"lod1.{city_name}_building_surfaces_lod1"

lod1_table_name = f"{city_name}_buildings_lod1"
lod1_surface_table_name = f"{city_name}_building_surfaces_lod1"

target_srid  = 4326
source_srid  = 3857

## 1.3 建表

In [11]:
# LOD1建筑主表
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod1_table_name_full} (
        building_id     VARCHAR PRIMARY KEY,
        block_id        VARCHAR,
        citygml_id      VARCHAR,
        geom_2d         GEOMETRY(Polygon, {target_srid}),
        height          FLOAT,
        floor_count     INTEGER,
        function        VARCHAR,
        ground_z        FLOAT
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod1_table_name}_geom_idx
    ON {lod1_table_name_full} USING GIST (geom_2d);
""", conn=conn)

# LOD1 surface子表（无citygml_id，面由计算生成）
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod1_surface_table_name_full} (
        surface_id      VARCHAR PRIMARY KEY,
        building_id     VARCHAR REFERENCES {lod1_table_name_full}(building_id),
        surface_type    VARCHAR,
        geom_3d         GEOMETRY(PolygonZ, {target_srid})
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod1_surface_table_name}_building_idx
    ON {lod1_surface_table_name_full} (building_id);
""", conn=conn)

print(city_prefix + " LOD1表创建完成")

CA_TO LOD1表创建完成


In [ ]:
# 清空building和surface表，用于需要重新处理的情况
utils_z.run_sql(f"TRUNCATE TABLE {lod1_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_table_name_full}表已清空")
utils_z.run_sql(f"TRUNCATE TABLE {lod1_surface_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_surface_table_name_full}表已清空")

In [24]:
utils_z.run_sql(f"TRUNCATE TABLE {lod1_surface_table_name_full} CASCADE;", conn=conn)

## 1.4 入库

In [13]:
shp_file_name = "3DMassingShapefile_2025_WGS84.shp"
shp_path = rf"E:\2_data\building_3d_opensource\{city_name}\shp\{shp_file_name}"

In [14]:
import traceback
conn.rollback()

In [16]:
print(f"读取SHP文件：{shp_path}")
gdf = gpd.read_file(shp_path)
print(f"总行数：{len(gdf)}")

# 获取当前最大ID，设置计数器初始值
cur = conn.cursor()

cur.execute(f"SELECT MAX(building_id) FROM {lod1_table_name_full}")
max_bid = cur.fetchone()[0]
building_counter = int(max_bid.split("_B_")[1]) + 1 if max_bid else 1
cur.close()

print(f"building_counter起始值：{building_counter}")

# 入库
try:
    count, building_counter = shpar.insert_buildings_shp_toronto(
        gdf, conn,
        lod1_table=lod1_table_name_full,
        city_prefix=city_prefix,
        building_counter=building_counter
    )
    print(f"\n完成！{city_name}共入库建筑：{count}")
except Exception as e:
    print(f"错误：{e}")
    traceback.print_exc()

读取SHP文件：E:\2_data\building_3d_opensource\toronto\shp\3DMassingShapefile_2025_WGS84.shp
总行数：428184
building_counter起始值：1
过滤后建筑数：409703

完成！toronto共入库建筑：409703


# 2. 叠合、建面

## 2.1 创建surface入库

In [ ]:
# 先查当前surface表最大编号
cur = conn.cursor()
cur.execute(f"SELECT MAX(surface_id) FROM {lod1_surface_table_name_full}")
max_sid = cur.fetchone()[0]
base = int(max_sid.split("_S_")[1]) if max_sid else 0
cur.close()

# GroundSurface
utils_z.run_sql(f"""
    INSERT INTO {lod1_surface_table_name_full} (surface_id, building_id, surface_type, geom_3d)
    SELECT 
        '{city_prefix}_S_' || LPAD((ROW_NUMBER() OVER () + {base})::TEXT, 8, '0'),
        building_id,
        'GroundSurface',
        ST_Force3D(ST_Translate(geom_2d, 0, 0, ground_z))
    FROM {lod1_table_name_full};
""", conn=conn)

# 更新base
cur = conn.cursor()
cur.execute(f"SELECT MAX(surface_id) FROM {lod1_surface_table_name_full}")
max_sid = cur.fetchone()[0]
base = int(max_sid.split("_S_")[1]) if max_sid else 0
cur.close()

# RoofSurface
utils_z.run_sql(f"""
    INSERT INTO {lod1_surface_table_name_full} (surface_id, building_id, surface_type, geom_3d)
    SELECT 
        '{city_prefix}_S_' || LPAD((ROW_NUMBER() OVER () + {base})::TEXT, 8, '0'),
        building_id,
        'RoofSurface',
        ST_Force3D(ST_Translate(geom_2d, 0, 0, ground_z + height))
    FROM {lod1_table_name_full};
""", conn=conn)

# 更新base
cur = conn.cursor()
cur.execute(f"SELECT MAX(surface_id) FROM {lod1_surface_table_name_full}")
max_sid = cur.fetchone()[0]
base = int(max_sid.split("_S_")[1]) if max_sid else 0
cur.close()

# WallSurface
utils_z.run_sql(f"""
    INSERT INTO {lod1_surface_table_name_full} (surface_id, building_id, surface_type, geom_3d)
    SELECT
        '{city_prefix}_S_' || LPAD((ROW_NUMBER() OVER () + {base})::TEXT, 8, '0'),
        building_id,
        'WallSurface',
        ST_MakePolygon(ST_MakeLine(ARRAY[
            ST_MakePoint(ST_X(p1), ST_Y(p1), ground_z),
            ST_MakePoint(ST_X(p2), ST_Y(p2), ground_z),
            ST_MakePoint(ST_X(p2), ST_Y(p2), ground_z + height),
            ST_MakePoint(ST_X(p1), ST_Y(p1), ground_z + height),
            ST_MakePoint(ST_X(p1), ST_Y(p1), ground_z)
        ]))
    FROM (
        SELECT
            b.building_id,
            b.ground_z,
            b.height,
            ST_PointN(ST_ExteriorRing(b.geom_2d), n) AS p1,
            ST_PointN(ST_ExteriorRing(b.geom_2d), n + 1) AS p2
        FROM {lod1_table_name_full} b,
        GENERATE_SERIES(1, ST_NPoints(ST_ExteriorRing(b.geom_2d)) - 1) AS n
    ) edges;
""", conn=conn)

# 统计
cur = conn.cursor()
cur.execute(f"""
    SELECT surface_type, COUNT(*) 
    FROM {lod1_surface_table_name_full} 
    GROUP BY surface_type ORDER BY surface_type
""")
rows = cur.fetchall()
for row in rows:
    print(f"  {row[0]}: {row[1]}")

# 计算总数
total = sum(row[1] for row in rows)
print(f"  Total: {total}")
cur.close()

  GroundSurface: 409703
  RoofSurface: 409703
  WallSurface: 4704452


In [ ]:
shpar.generate_surfaces_from_buildings(
    conn,
    lod1_table=lod1_table_name_full,
    surface_table=lod1_surface_table_name_full,
    city_prefix=city_prefix
)

# 统计
cur = conn.cursor()
cur.execute(f"""
    SELECT surface_type, COUNT(*) 
    FROM {lod1_surface_table_name_full} 
    GROUP BY surface_type ORDER BY surface_type
""")
rows = cur.fetchall()
for row in rows:
    print(f"  {row[0]}: {row[1]}")

# 计算总数
total = sum(row[1] for row in rows)
print(f"  Total: {total}")
cur.close()

## 2.2 与block叠合

In [18]:
# 确认空间索引
print(block_table_name, lod1_table_name_full)
utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod1_table_name}_geom_idx
    ON {lod1_table_name_full} USING GIST (geom_2d);
""", conn=conn)
print("准备开始空间叠合...")

block.toronto_blocks lod1.toronto_buildings_lod1
准备开始空间叠合...


In [19]:
utils_z.run_sql(f"""
    UPDATE {lod1_table_name_full} b
    SET block_id = (
        SELECT bl.block_id
        FROM {block_table_name} bl
        WHERE ST_Within(ST_Centroid(b.geom_2d), bl.geom)
        LIMIT 1
    );
""", conn=conn)
print("空间叠合完成")

空间叠合完成


In [21]:
# 检查匹配情况
result = utils_z.run_sql(f"""
    SELECT
        COUNT(*) AS total,
        COUNT(block_id) AS matched,
        COUNT(*) FILTER (WHERE block_id IS NULL) AS unmatched
    FROM {lod1_table_name_full};
""", conn=conn, fetch=True)

print(f"总建筑数：{result[0][0]}")
print(f"成功匹配block：{result[0][1]}")
print(f"未匹配block：{result[0][2]}")

# 检查包含LOD1建筑的街区数量
sql_counts = f"SELECT COUNT(DISTINCT block_id) FROM {lod1_table_name_full} WHERE block_id IS NOT NULL;"
(lod1_count,) = utils_z.run_sql(sql_counts, conn=conn, fetch=True)[0]

print(f"包含 LOD1 建筑的街区数量: {lod1_count}")

总建筑数：409703
成功匹配block：404248
未匹配block：5455
包含 LOD1 建筑的街区数量: 10107
